# DeepSeek-R1-Distill-Qwen-14B + LoRA — Test Inferenta

Notebook de testare: incarca modelul DeepSeek-R1-Distill-Qwen-14B,
aplica adaptoarele LoRA antrenate pe exercitii BAC, si testeaza inferenta.
Modelul foloseste format `<think>...</think>` pentru chain-of-thought reasoning.

## Setup Kaggle
1. **GPU**: Settings → Accelerator → GPU T4 x2
2. **Internet**: Settings → Internet → On
3. Adaptorul LoRA se incarca din Output-ul notebook-ului de antrenare (sau HuggingFace)

In [ ]:
# Celula 1: Instalare dependinte
!pip install -q transformers>=4.40.0 peft>=0.10.0 accelerate>=0.30.0 \
    bitsandbytes>=0.46.1 sentencepiece huggingface_hub

In [ ]:
# Celula 2: Configurare + incarcare adapter
import torch
import os
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

# ==============================
# CONFIGUREAZA AICI
# ==============================

# Modelul base — se descarca automat de pe HuggingFace
BASE_MODEL = "deepseek-ai/DeepSeek-R1-Distill-Qwen-14B"

# Adaptor LoRA — ALEGE una din optiuni:

# Optiunea 1: Din Output-ul notebook-ului de antrenare (daca sunt in acelasi proiect Kaggle)
ADAPTER_PATH = "/kaggle/input/smartbac-lora-final"

# Optiunea 2: Din HuggingFace
# from huggingface_hub import snapshot_download, login
# HF_REPO = "USERNAME/smartbac-deepseek-lora"
# HF_TOKEN = "hf_XXXXXXXXXXXXXXXXXXXX"
# login(token=HF_TOKEN)
# ADAPTER_PATH = "/kaggle/working/adapter"
# if not os.path.exists(ADAPTER_PATH):
#     snapshot_download(repo_id=HF_REPO, local_dir=ADAPTER_PATH, token=HF_TOKEN)

# Cauta adaptorul in locatii comune daca nu exista
if not os.path.exists(ADAPTER_PATH):
    for p in [
        "/kaggle/input/smartbac-lora-final",
        "/kaggle/input/smartbac-lora/smartbac-lora-final",
        "/kaggle/working/smartbac-lora-final",
    ]:
        if os.path.exists(p):
            ADAPTER_PATH = p
            break

print(f"Model base: {BASE_MODEL}")
print(f"Adapter LoRA: {ADAPTER_PATH} (exists: {os.path.exists(ADAPTER_PATH)})")

if os.path.exists(ADAPTER_PATH):
    print(f"\nFisiere adapter:")
    for f in sorted(os.listdir(ADAPTER_PATH)):
        fpath = os.path.join(ADAPTER_PATH, f)
        if os.path.isfile(fpath):
            print(f"  {f} ({os.path.getsize(fpath) / 1e6:.1f} MB)")

In [ ]:
# Celula 3: Incarcare model base cu quantizare 4-bit
print(f"Incarc {BASE_MODEL} (4-bit)...")

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
    torch_dtype=torch.float16,
)

tokenizer = AutoTokenizer.from_pretrained(
    BASE_MODEL,
    trust_remote_code=True,
    padding_side="right",
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    model.config.pad_token_id = tokenizer.eos_token_id

print(f"Model base incarcat: {model.num_parameters():,} parametri")
print(f"VRAM: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

In [ ]:
# Celula 4: Aplicare adaptoare LoRA antrenate
print(f'Aplic adaptoarele LoRA din: {ADAPTER_PATH}')

model = PeftModel.from_pretrained(
    model,
    ADAPTER_PATH,
    is_trainable=False,  # Doar inferenta, nu antrenament
)

model.eval()

# Afiseaza info despre adaptoare
model.print_trainable_parameters()
print(f'\nVRAM dupa LoRA: {torch.cuda.memory_allocated() / 1e9:.2f} GB')
print('\nModel GATA pentru inferenta!')

In [ ]:
# Celula 5: Functie de inferenta cu <think> parsing

SYSTEM_PROMPT = (
    "Ești SmartBAC, un asistent de matematică specializat pe exerciții de Bacalaureat. "
    "Gândește pas cu pas în blocul <think>, apoi dă răspunsul final. "
    "Răspunde în limba română."
)


def solve(question, max_new_tokens=1024, temperature=0.1, top_k=30):
    """Rezolva un exercitiu BAC folosind modelul fine-tuned cu <think> reasoning."""
    prompt = (
        f"<|im_start|>system\n{SYSTEM_PROMPT}\n<|im_end|>\n"
        f"<|im_start|>user\n{question}\n<|im_end|>\n"
        f"<|im_start|>assistant\n"
    )
    
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            top_k=top_k,
            do_sample=True,
            pad_token_id=tokenizer.pad_token_id,
            repetition_penalty=1.1,
        )
    
    response = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=False,
    )
    
    # Taie la <|im_end|>
    if "<|im_end|>" in response:
        response = response[:response.index("<|im_end|>")]
    
    return response.strip()


def solve_pretty(question, **kwargs):
    """Rezolva si afiseaza frumos cu thinking separat."""
    raw = solve(question, **kwargs)
    
    if "</think>" in raw:
        think_part = raw.split("</think>")[0].replace("<think>", "").strip()
        answer_part = raw.split("</think>")[1].strip()
        print(f"🧠 Thinking:")
        print(f"   {think_part}")
        print(f"\n💡 Răspuns:")
        print(f"   {answer_part}")
    else:
        print(f"💡 {raw}")
    
    return raw


print("Functii definite: solve() si solve_pretty()")

In [ ]:
# Celula 6: TEST INFERENTA — Exercitii BAC

test_exercises = [
    # Ecuatii
    "Rezolvați ecuația: 2x + 3 = 7",
    "Rezolvați ecuația: x^2 - 5x + 6 = 0",
    
    # Derivate
    "Calculați derivata funcției f(x) = x^3 - 6x^2 + 9x + 1",
    
    # Limite
    "Calculați limita: lim(x->inf) (x^2 + 3x) / (2x^2 - 1)",
    
    # Matrice
    "Calculați determinantul matricei A = [[3, 1], [2, 4]]",
    
    # Integrale
    "Calculați integrala: ∫(2x + 1)dx",
    
    # Probabilitati
    "Se aruncă un zar de 2 ori. Care este probabilitatea ca suma să fie 7?",
    
    # Combinatorica
    "Câte numere de 3 cifre distincte se pot forma cu cifrele 1, 2, 3, 4, 5?",
]

print("=" * 70)
print("  TEST INFERENTA — DeepSeek-R1 + LoRA BAC (<think> reasoning)")
print("=" * 70)

for i, q in enumerate(test_exercises, 1):
    print(f'\n{"─" * 70}')
    print(f"  [{i}/{len(test_exercises)}] {q}")
    print(f'{"─" * 70}')
    
    solve_pretty(q)

print(f'\n{"=" * 70}')
print("  TEST COMPLET!")
print(f'{"=" * 70}')

In [ ]:
# Celula 7: Comparatie MODEL BASE vs MODEL FINE-TUNED

print("Comparatie: Model Base vs Model Fine-tuned (LoRA)")
print("=" * 70)

compare_questions = [
    "Rezolvați ecuația: x^2 - 4 = 0",
    "Calculați derivata funcției f(x) = ln(x^2 + 1)",
]

# Raspunsuri cu LoRA (fine-tuned)
print("\n>>> CU LoRA (fine-tuned pe BAC):")
for q in compare_questions:
    print(f"\nQ: {q}")
    solve_pretty(q)

# Dezactiveaza adaptoarele temporar
model.disable_adapter_layers()

print(f'\n{"=" * 70}')
print("\n>>> FARA LoRA (model base DeepSeek-R1):")
for q in compare_questions:
    print(f"\nQ: {q}")
    solve_pretty(q)

# Reactiveaza adaptoarele
model.enable_adapter_layers()
print(f'\n{"=" * 70}')
print("Adaptoarele LoRA au fost reactivate.")

In [ ]:
# Celula 8: Test interactiv — scrie propriul exercitiu
# Modifica textul si ruleaza celula!

exercitiu = "Determinați ecuația tangentei la graficul funcției f(x) = x^2 - 3x + 2 în punctul de abscisă x=1"

print(f"Exercitiu: {exercitiu}")
print(f'{"─" * 70}')
solve_pretty(exercitiu, max_new_tokens=1024, temperature=0.1)